In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# パターン選択（1つだけTrueにする）
# ============================================================
PATTERN_A_REGULARIZE = True    # 正則化強化（特徴量は元コードのまま）
PATTERN_B_DROP_SNV   = False   # SNV除外（d1のみ）

if PATTERN_A_REGULARIZE:
    pattern_name = "正則化強化"
elif PATTERN_B_DROP_SNV:
    pattern_name = "SNV除外"
else:
    pattern_name = "元コード再現"

print("=" * 60)
print(f"🧪 テスト: {pattern_name}")
if PATTERN_A_REGULARIZE:
    print("  変更点: colsample 0.3→0.2, min_child 20→30, reg_lambda 0→1.0")
elif PATTERN_B_DROP_SNV:
    print("  変更点: SNVをLGB入力から除外 (3123→1568次元)")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

# ============================================================
# 2. CVループ
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
oof_lgb = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── 前処理（元コードと同一）──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)

    d1_tr = savgol_filter(snv_tr, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_va = savgol_filter(snv_va, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_te = savgol_filter(snv_te, window_length=15, polyorder=2, deriv=1, axis=1)

    ratio_tr = (X_tr_raw[:, idx_1940] / (X_tr_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_va = (X_va_raw[:, idx_1940] / (X_va_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_te = (X_te_raw[:, idx_1940] / (X_te_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)

    std_tr = np.std(X_tr_raw, axis=1, keepdims=True)
    std_va = np.std(X_va_raw, axis=1, keepdims=True)
    std_te = np.std(X_te_raw, axis=1, keepdims=True)

    # ── PCA + KNN（元コードと同一）──
    pca = PCA(n_components=10, random_state=42)
    pca_tr = pca.fit_transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_tr)

    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)

    _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── LGB入力の組み立て ──
    if PATTERN_B_DROP_SNV:
        # SNV除外: d1 + PCA + KNN + ratio + std
        feat_tr = np.hstack([d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
        feat_va = np.hstack([d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
        feat_te = np.hstack([d1_te, pca_te, knn_ymean_te, ratio_te, std_te])
    else:
        # 元コードと同一: SNV + d1 + PCA + KNN + ratio + std
        feat_tr = np.hstack([snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
        feat_va = np.hstack([snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
        feat_te = np.hstack([snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te])

    if fold == 0:
        print(f"\n  📐 LGB入力次元: {feat_tr.shape[1]}")

    # ── LightGBM ──
    if PATTERN_A_REGULARIZE:
        lgb_model = lgb.LGBMRegressor(
            n_estimators=1500,       # 増やす（正則化で遅く学習するため）
            learning_rate=0.02,      # 少し下げる
            max_depth=4,             # 5→4（浅い木=一般的パターン）
            num_leaves=15,           # 31→15（単純な分岐）
            subsample=0.7,           # 0.8→0.7（データのサブサンプル）
            colsample_bytree=0.2,    # 0.3→0.2（特徴量の使用を制限）
            min_child_samples=30,    # default→30（葉に必要なサンプル数増加）
            reg_alpha=0.1,           # L1正則化
            reg_lambda=1.0,          # L2正則化
            random_state=42, verbosity=-1
        )
    else:
        lgb_model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
        )

    lgb_model.fit(
        feat_tr, y_tr,
        eval_set=[(feat_va, y_va)],
        callbacks=[lgb.early_stopping(50 if PATTERN_A_REGULARIZE else 30, verbose=False)]
    )
    p_va = np.expm1(lgb_model.predict(feat_va))
    p_te = np.expm1(lgb_model.predict(feat_te))

    oof_lgb[va_idx] = p_va
    final_lgb += p_te / 5

    y_va_real = np.expm1(y_va)
    rmse = np.sqrt(mean_squared_error(y_va_real, p_va))
    fold_rmses.append(rmse)
    print(f"  🌟 LGB RMSE: {rmse:.4f}")

    # 特徴量重要度（Fold 0）
    if fold == 0:
        imp = lgb_model.feature_importances_
        cat = {}
        pos = 0
        if not PATTERN_B_DROP_SNV:
            n_snv = snv_tr.shape[1]
            cat['SNV'] = np.sum(imp[pos:pos+n_snv]); pos += n_snv
        n_d1 = d1_tr.shape[1]
        cat['d1'] = np.sum(imp[pos:pos+n_d1]); pos += n_d1
        cat['PCA'] = np.sum(imp[pos:pos+10]); pos += 10
        cat['KNN_mean'] = imp[pos]; pos += 1
        cat['ratio'] = imp[pos]; pos += 1
        cat['std'] = imp[pos]; pos += 1

        total = sum(cat.values())
        print(f"\n  📊 特徴量重要度:")
        for name, val in sorted(cat.items(), key=lambda x: -x[1]):
            pct = val / total * 100
            bar = '█' * int(pct / 2)
            print(f"     {name:15s}: {val:6.0f} ({pct:5.1f}%) {bar}")


# ============================================================
# 3. 全体評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_lgb))

print(f"  🌟 LGB OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# 樹種別
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_lgb[mask.values]
    rmse_s = np.sqrt(np.mean((y_s - p_s)**2))
    bias_s = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_s:7.2f} {bias_s:+7.2f}")


# ============================================================
# 4. 提出ファイル
# ============================================================
final_out = np.clip(final_lgb, 0, None)

submit[1] = final_out
out = f'submission_{pattern_name}.csv'
submit.to_csv(out, index=False, header=False)

print(f"\n✅ 提出ファイル: {out}")
print(f"📈 min={final_out.min():.1f}%, median={np.median(final_out):.1f}%, "
      f"max={final_out.max():.1f}%")

print(f"\n📌 スコア比較:")
print(f"   LGB単独(元特徴量):  LB = 12.615 ← 現BEST")
print(f"   元Blend:           LB = 12.647")
print(f"   LGB+加重KNN:       LB = 12.940 ← 悪化")
print(f"   LGB+d2:            LB = 13.410 ← 悪化")
print(f"   今回({pattern_name}):  LB = ???")

🧪 テスト: 正則化強化
  変更点: colsample 0.3→0.2, min_child 20→30, reg_lambda 0→1.0

───────────────────────────────────────────────────────
📁 Fold 1/5  (train:940, valid:270)
   検証樹種: ['ウエンジ', 'トチ']

  📐 LGB入力次元: 3123
  🌟 LGB RMSE: 9.1516

  📊 特徴量重要度:
     d1             :   1755 ( 61.9%) ██████████████████████████████
     SNV            :    724 ( 25.6%) ████████████
     KNN_mean       :    255 (  9.0%) ████
     PCA            :     80 (  2.8%) █
     ratio          :     18 (  0.6%) 
     std            :      1 (  0.0%) 

───────────────────────────────────────────────────────
📁 Fold 2/5  (train:981, valid:229)
   検証樹種: ['チェリー', 'ヒノキ']
  🌟 LGB RMSE: 19.4523

───────────────────────────────────────────────────────
📁 Fold 3/5  (train:1009, valid:201)
   検証樹種: ['ウォールナット', 'クリ']
  🌟 LGB RMSE: 21.4418

───────────────────────────────────────────────────────
📁 Fold 4/5  (train:959, valid:251)
   検証樹種: ['ナラ', 'ベイマツ', 'ホワイトオーク']
  🌟 LGB RMSE: 18.9477

──────────────────────────────────────────────

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# テスト選択（1つだけTrueにする）
# ============================================================
TEST_HUBER         = True    # 損失関数をHuberに変更（他は元コード同一）
TEST_DROP_SNV      = False   # SNV除外（3123→1568次元）
TEST_HUBER_DROP    = False   # Huber + SNV除外の組み合わせ

if TEST_HUBER_DROP:
    pattern_name = "Huber + SNV除外"
elif TEST_HUBER:
    pattern_name = "Huber Loss"
elif TEST_DROP_SNV:
    pattern_name = "SNV除外"
else:
    pattern_name = "元コード再現"

use_huber = TEST_HUBER or TEST_HUBER_DROP
drop_snv = TEST_DROP_SNV or TEST_HUBER_DROP

print("=" * 60)
print(f"🧪 テスト: {pattern_name}")
if use_huber:
    print("  → 損失関数: MSE → Huber（大きな残差にロバスト）")
if drop_snv:
    print("  → SNVをLGB入力から除外（1555次元削減）")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

# ============================================================
# 2. CVループ
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
oof_lgb = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── 前処理（元コードと同一）──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)

    d1_tr = savgol_filter(snv_tr, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_va = savgol_filter(snv_va, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_te = savgol_filter(snv_te, window_length=15, polyorder=2, deriv=1, axis=1)

    ratio_tr = (X_tr_raw[:, idx_1940] / (X_tr_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_va = (X_va_raw[:, idx_1940] / (X_va_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_te = (X_te_raw[:, idx_1940] / (X_te_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)

    std_tr = np.std(X_tr_raw, axis=1, keepdims=True)
    std_va = np.std(X_va_raw, axis=1, keepdims=True)
    std_te = np.std(X_te_raw, axis=1, keepdims=True)

    # ── PCA + KNN（元コードと同一）──
    pca = PCA(n_components=10, random_state=42)
    pca_tr = pca.fit_transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_tr)

    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)

    _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── LGB入力 ──
    if drop_snv:
        feat_tr = np.hstack([d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
        feat_va = np.hstack([d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
        feat_te = np.hstack([d1_te, pca_te, knn_ymean_te, ratio_te, std_te])
    else:
        feat_tr = np.hstack([snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
        feat_va = np.hstack([snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
        feat_te = np.hstack([snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te])

    if fold == 0:
        print(f"\n  📐 LGB入力次元: {feat_tr.shape[1]}")

    # ── LightGBM（Huber or 元コード）──
    if use_huber:
        lgb_model = lgb.LGBMRegressor(
            objective='huber',          # ← 唯一の変更点
            n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
        )
    else:
        lgb_model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
        )

    lgb_model.fit(
        feat_tr, y_tr,
        eval_set=[(feat_va, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va = np.expm1(lgb_model.predict(feat_va))
    p_te = np.expm1(lgb_model.predict(feat_te))

    oof_lgb[va_idx] = p_va
    final_lgb += p_te / 5

    y_va_real = np.expm1(y_va)
    rmse = np.sqrt(mean_squared_error(y_va_real, p_va))
    fold_rmses.append(rmse)
    print(f"  🌟 LGB RMSE: {rmse:.4f}")

    # 特徴量重要度（Fold 0）
    if fold == 0:
        imp = lgb_model.feature_importances_
        cat = {}
        pos = 0
        if not drop_snv:
            n_snv = snv_tr.shape[1]
            cat['SNV'] = np.sum(imp[pos:pos+n_snv]); pos += n_snv
        n_d1 = d1_tr.shape[1]
        cat['d1'] = np.sum(imp[pos:pos+n_d1]); pos += n_d1
        cat['PCA'] = np.sum(imp[pos:pos+10]); pos += 10
        cat['KNN_mean'] = imp[pos]; pos += 1
        cat['ratio'] = imp[pos]; pos += 1
        cat['std'] = imp[pos]; pos += 1

        total = sum(cat.values())
        print(f"\n  📊 特徴量重要度:")
        for name, val in sorted(cat.items(), key=lambda x: -x[1]):
            pct = val / total * 100
            bar = '█' * int(pct / 2)
            print(f"     {name:15s}: {val:6.0f} ({pct:5.1f}%) {bar}")


# ============================================================
# 3. 全体評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_lgb))

print(f"  🌟 LGB OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# 樹種別
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_lgb[mask.values]
    rmse_s = np.sqrt(np.mean((y_s - p_s)**2))
    bias_s = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_s:7.2f} {bias_s:+7.2f}")


# ============================================================
# 4. 提出ファイル
# ============================================================
final_out = np.clip(final_lgb, 0, None)
submit[1] = final_out
out = f'submission_{pattern_name.replace(" ", "_")}.csv'
submit.to_csv(out, index=False, header=False)

print(f"\n✅ 提出ファイル: {out}")
print(f"📈 min={final_out.min():.1f}%, median={np.median(final_out):.1f}%, "
      f"max={final_out.max():.1f}%")

print(f"\n📌 スコア比較:")
print(f"   LGB単独(元):       LB = 12.615 ← 現BEST")
print(f"   元Blend:           LB = 12.647")
print(f"   正則化強化:         LB = 12.760")
print(f"   LGB+加重KNN:       LB = 12.940")
print(f"   LGB+d2:            LB = 13.410")
print(f"   今回({pattern_name}):  LB = ???")

🧪 テスト: Huber Loss
  → 損失関数: MSE → Huber（大きな残差にロバスト）

───────────────────────────────────────────────────────
📁 Fold 1/5  (train:940, valid:270)
   検証樹種: ['ウエンジ', 'トチ']

  📐 LGB入力次元: 3123
  🌟 LGB RMSE: 9.1639

  📊 特徴量重要度:
     d1             :   2173 ( 59.2%) █████████████████████████████
     SNV            :   1045 ( 28.5%) ██████████████
     KNN_mean       :    331 (  9.0%) ████
     PCA            :     83 (  2.3%) █
     ratio          :     32 (  0.9%) 
     std            :      4 (  0.1%) 

───────────────────────────────────────────────────────
📁 Fold 2/5  (train:981, valid:229)
   検証樹種: ['チェリー', 'ヒノキ']
  🌟 LGB RMSE: 19.4496

───────────────────────────────────────────────────────
📁 Fold 3/5  (train:1009, valid:201)
   検証樹種: ['ウォールナット', 'クリ']
  🌟 LGB RMSE: 21.2133

───────────────────────────────────────────────────────
📁 Fold 4/5  (train:959, valid:251)
   検証樹種: ['ナラ', 'ベイマツ', 'ホワイトオーク']
  🌟 LGB RMSE: 18.5891

───────────────────────────────────────────────────────
📁 Fold 5/5

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# テスト選択（1つだけTrueにする）
# ============================================================
TEST_DROP_SNV      = False    # SNV除外（3123→1568次元）
TEST_KNN_K3        = False   # KNN k=5→3
TEST_PLS_AS_FEAT   = True   # PLS予測をLGBの特徴量に

if TEST_DROP_SNV:
    pattern_name = "SNV除外"
elif TEST_KNN_K3:
    pattern_name = "KNN_k=3"
elif TEST_PLS_AS_FEAT:
    pattern_name = "PLS予測を特徴量に"
else:
    pattern_name = "元コード再現"

print("=" * 60)
print(f"🧪 テスト: {pattern_name}")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

# ============================================================
# 2. CVループ
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
oof_lgb = np.zeros(len(train))
fold_rmses = []

# KNNのk値
knn_k = 3 if TEST_KNN_K3 else 5

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── 前処理 ──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)

    d1_tr = savgol_filter(snv_tr, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_va = savgol_filter(snv_va, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_te = savgol_filter(snv_te, window_length=15, polyorder=2, deriv=1, axis=1)

    ratio_tr = (X_tr_raw[:, idx_1940] / (X_tr_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_va = (X_va_raw[:, idx_1940] / (X_va_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_te = (X_te_raw[:, idx_1940] / (X_te_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)

    std_tr = np.std(X_tr_raw, axis=1, keepdims=True)
    std_va = np.std(X_va_raw, axis=1, keepdims=True)
    std_te = np.std(X_te_raw, axis=1, keepdims=True)

    # ── PCA + KNN ──
    pca = PCA(n_components=10, random_state=42)
    pca_tr = pca.fit_transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    knn = NearestNeighbors(n_neighbors=knn_k, metric='cosine')
    knn.fit(pca_tr)

    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=knn_k + 1)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    _, ind_va = knn.kneighbors(pca_va, n_neighbors=knn_k)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)

    _, ind_te = knn.kneighbors(pca_te, n_neighbors=knn_k)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── PLS予測を特徴量にする場合 ──
    if TEST_PLS_AS_FEAT:
        d2_tr = savgol_filter(snv_tr, window_length=11, polyorder=2, deriv=2, axis=1)
        d2_va = savgol_filter(snv_va, window_length=11, polyorder=2, deriv=2, axis=1)
        d2_te = savgol_filter(snv_te, window_length=11, polyorder=2, deriv=2, axis=1)

        pls_model = PLSRegression(n_components=7)
        pls_model.fit(d2_tr, y_tr)
        pls_pred_tr = pls_model.predict(d2_tr).flatten().reshape(-1, 1)
        pls_pred_va = pls_model.predict(d2_va).flatten().reshape(-1, 1)
        pls_pred_te = pls_model.predict(d2_te).flatten().reshape(-1, 1)

    # ── LGB入力の組み立て ──
    if TEST_DROP_SNV:
        parts_tr = [d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr]
        parts_va = [d1_va, pca_va, knn_ymean_va, ratio_va, std_va]
        parts_te = [d1_te, pca_te, knn_ymean_te, ratio_te, std_te]
    else:
        parts_tr = [snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr]
        parts_va = [snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va]
        parts_te = [snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te]

    if TEST_PLS_AS_FEAT:
        parts_tr.append(pls_pred_tr)
        parts_va.append(pls_pred_va)
        parts_te.append(pls_pred_te)

    feat_tr = np.hstack(parts_tr)
    feat_va = np.hstack(parts_va)
    feat_te = np.hstack(parts_te)

    if fold == 0:
        print(f"\n  📐 LGB入力次元: {feat_tr.shape[1]}")
        if TEST_DROP_SNV:
            print(f"     d1({d1_tr.shape[1]}) + PCA(10) + KNN(1) + ratio(1) + std(1)")
        elif TEST_PLS_AS_FEAT:
            print(f"     SNV + d1 + PCA + KNN + ratio + std + PLS予測(1)")
        else:
            print(f"     SNV + d1 + PCA + KNN(k={knn_k}) + ratio + std")

    # ── LightGBM（元コードと同一パラメータ）──
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr, y_tr,
        eval_set=[(feat_va, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va = np.expm1(lgb_model.predict(feat_va))
    p_te = np.expm1(lgb_model.predict(feat_te))

    oof_lgb[va_idx] = p_va
    final_lgb += p_te / 5

    y_va_real = np.expm1(y_va)
    rmse = np.sqrt(mean_squared_error(y_va_real, p_va))
    fold_rmses.append(rmse)
    print(f"  🌟 LGB RMSE: {rmse:.4f}")

    # 特徴量重要度（Fold 0）
    if fold == 0:
        imp = lgb_model.feature_importances_
        cat = {}
        pos = 0
        if not TEST_DROP_SNV:
            cat['SNV'] = np.sum(imp[pos:pos+snv_tr.shape[1]]); pos += snv_tr.shape[1]
        cat['d1'] = np.sum(imp[pos:pos+d1_tr.shape[1]]); pos += d1_tr.shape[1]
        cat['PCA'] = np.sum(imp[pos:pos+10]); pos += 10
        cat['KNN_mean'] = imp[pos]; pos += 1
        cat['ratio'] = imp[pos]; pos += 1
        cat['std'] = imp[pos]; pos += 1
        if TEST_PLS_AS_FEAT:
            cat['PLS_pred'] = imp[pos]; pos += 1

        total = sum(cat.values())
        print(f"\n  📊 特徴量重要度:")
        for name, val in sorted(cat.items(), key=lambda x: -x[1]):
            pct = val / total * 100
            bar = '█' * int(pct / 2)
            print(f"     {name:15s}: {val:6.0f} ({pct:5.1f}%) {bar}")


# ============================================================
# 3. 全体評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_lgb))

print(f"  🌟 LGB OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# 樹種別
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_lgb[mask.values]
    rmse_s = np.sqrt(np.mean((y_s - p_s)**2))
    bias_s = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_s:7.2f} {bias_s:+7.2f}")


# ============================================================
# 4. 提出ファイル
# ============================================================
final_out = np.clip(final_lgb, 0, None)
submit[1] = final_out
out = f'submission_{pattern_name.replace(" ", "_")}.csv'
submit.to_csv(out, index=False, header=False)

print(f"\n✅ 提出ファイル: {out}")
print(f"📈 min={final_out.min():.1f}%, median={np.median(final_out):.1f}%, "
      f"max={final_out.max():.1f}%")

print(f"\n📌 全スコア比較:")
print(f"   LGB単独(元):       LB = 12.615 ← 現BEST")
print(f"   元Blend:           LB = 12.647")
print(f"   正則化強化:         LB = 12.760")
print(f"   Huber Loss:        LB = 12.770")
print(f"   LGB+加重KNN:       LB = 12.940")
print(f"   LGB+d2:            LB = 13.410")
print(f"   今回({pattern_name}):  LB = ???")

🧪 テスト: PLS予測を特徴量に

───────────────────────────────────────────────────────
📁 Fold 1/5  (train:940, valid:270)
   検証樹種: ['ウエンジ', 'トチ']

  📐 LGB入力次元: 3124
     SNV + d1 + PCA + KNN + ratio + std + PLS予測(1)
  🌟 LGB RMSE: 9.1872

  📊 特徴量重要度:
     d1             :   2240 ( 55.8%) ███████████████████████████
     SNV            :    964 ( 24.0%) ███████████
     KNN_mean       :    445 ( 11.1%) █████
     PLS_pred       :    254 (  6.3%) ███
     PCA            :     87 (  2.2%) █
     ratio          :     25 (  0.6%) 
     std            :      2 (  0.0%) 

───────────────────────────────────────────────────────
📁 Fold 2/5  (train:981, valid:229)
   検証樹種: ['チェリー', 'ヒノキ']
  🌟 LGB RMSE: 18.1791

───────────────────────────────────────────────────────
📁 Fold 3/5  (train:1009, valid:201)
   検証樹種: ['ウォールナット', 'クリ']
  🌟 LGB RMSE: 17.0241

───────────────────────────────────────────────────────
📁 Fold 4/5  (train:959, valid:251)
   検証樹種: ['ナラ', 'ベイマツ', 'ホワイトオーク']
  🌟 LGB RMSE: 16.3863

───────────

In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# テスト選択
# ============================================================
TEST_MIXUP           = True    # Mixup augmentation
TEST_NOISE           = False   # ノイズ注入のみ
TEST_MIXUP_AND_NOISE = False   # 両方

if TEST_MIXUP_AND_NOISE:
    pattern_name = "Mixup + Noise"
elif TEST_MIXUP:
    pattern_name = "Mixup"
elif TEST_NOISE:
    pattern_name = "Noise注入"
else:
    pattern_name = "元コード再現"

use_mixup = TEST_MIXUP or TEST_MIXUP_AND_NOISE
use_noise = TEST_NOISE or TEST_MIXUP_AND_NOISE

print("=" * 60)
print(f"🧪 テスト: {pattern_name}")
if use_mixup:
    print("  → 異なる樹種のスペクトルを混合して疑似データ生成")
if use_noise:
    print("  → 小さなガウスノイズで測定ばらつきを模擬")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 2. Augmentation関数
# ============================================================

def mixup_augmentation(X, y, species, n_augment=500, alpha=0.3, seed=42):
    """
    異なる樹種のサンプルを混合して疑似データを生成
    
    X: 生スペクトル (n_samples, n_wavelengths)
    y: log1p(含水率)
    species: 樹種番号
    n_augment: 生成するサンプル数
    alpha: Beta分布のパラメータ（小さいほど元データに近い混合）
    """
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    
    X_aug = []
    y_aug = []
    
    for _ in range(n_augment):
        # 異なる樹種から1サンプルずつ選ぶ
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        
        # 混合比率（Beta分布）
        lam = rng.beta(alpha, alpha)
        
        # スペクトルと含水率を線形補間
        x_mix = lam * X[idx1] + (1 - lam) * X[idx2]
        y_mix = lam * y[idx1] + (1 - lam) * y[idx2]
        
        X_aug.append(x_mix)
        y_aug.append(y_mix)
    
    return np.array(X_aug), np.array(y_aug)


def noise_augmentation(X, y, n_augment=300, noise_std=0.002, seed=42):
    """
    元データに小さなノイズを加えて疑似データを生成
    測定時のノイズ・プローブ距離変動を模擬
    """
    rng = np.random.RandomState(seed)
    
    indices = rng.choice(len(X), size=n_augment, replace=True)
    X_aug = X[indices] + rng.normal(0, noise_std, (n_augment, X.shape[1]))
    y_aug = y[indices]
    
    return X_aug, y_aug


# ============================================================
# 3. 特徴量作成関数（元コードと同一の処理をまとめる）
# ============================================================

def make_features(X_raw):
    """生スペクトルから元コードと同一の特徴量を作成"""
    snv = apply_snv(X_raw)
    d1 = savgol_filter(snv, window_length=15, polyorder=2, deriv=1, axis=1)
    ratio = (X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    std = np.std(X_raw, axis=1, keepdims=True)
    return snv, d1, ratio, std


# ============================================================
# 4. CVループ
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
oof_lgb = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    tr_species_nums = groups.iloc[tr_idx].values
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── Augmentation（生スペクトルレベルで実施）──
    X_tr_aug = X_tr_raw.copy()
    y_tr_aug = y_tr.copy()

    if use_mixup:
        X_mix, y_mix = mixup_augmentation(
            X_tr_raw, y_tr, tr_species_nums,
            n_augment=500, alpha=0.3, seed=42+fold
        )
        X_tr_aug = np.vstack([X_tr_aug, X_mix])
        y_tr_aug = np.concatenate([y_tr_aug, y_mix])

    if use_noise:
        X_noi, y_noi = noise_augmentation(
            X_tr_raw, y_tr,
            n_augment=300, noise_std=0.002, seed=42+fold
        )
        X_tr_aug = np.vstack([X_tr_aug, X_noi])
        y_tr_aug = np.concatenate([y_tr_aug, y_noi])

    if fold == 0:
        print(f"   元データ: {len(X_tr_raw)} samples")
        print(f"   拡張後:   {len(X_tr_aug)} samples (+{len(X_tr_aug)-len(X_tr_raw)})")

    # ── 特徴量作成（元コードと同一処理）──
    snv_tr, d1_tr, ratio_tr, std_tr = make_features(X_tr_aug)
    snv_va, d1_va, ratio_va, std_va = make_features(X_va_raw)
    snv_te, d1_te, ratio_te, std_te = make_features(X_te_raw)

    # ── PCA（元の訓練データのみでfit → 拡張データにtransform）──
    snv_tr_orig = apply_snv(X_tr_raw)  # PCA fitは元データのみ
    pca = PCA(n_components=10, random_state=42)
    pca.fit(snv_tr_orig)  # 元データのみでfit
    pca_tr = pca.transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    # ── KNN（元データのみでfit）──
    pca_tr_orig = pca.transform(snv_tr_orig)
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_tr_orig)

    # Train augmented
    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    # 元データ部分は自分除外、拡張部分はそのまま
    knn_ymean_tr = np.zeros(len(X_tr_aug))
    n_orig = len(X_tr_raw)
    for i in range(len(X_tr_aug)):
        neighbors = ind_tr[i]
        if i < n_orig:
            # 元データ: 自分を除外
            valid = neighbors[neighbors != i][:5]
        else:
            # 拡張データ: そのまま上位5つ
            valid = neighbors[:5]
        knn_ymean_tr[i] = np.mean(y_tr[:n_orig][valid])  # 元データのyのみ使用
    knn_ymean_tr = knn_ymean_tr.reshape(-1, 1)

    # Validation
    _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[:n_orig][ind_va], axis=1).reshape(-1, 1)

    # Test
    _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[:n_orig][ind_te], axis=1).reshape(-1, 1)

    # ── LGB入力（元コードと同一構成）──
    feat_tr = np.hstack([snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
    feat_va = np.hstack([snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
    feat_te = np.hstack([snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te])

    if fold == 0:
        print(f"   📐 LGB入力次元: {feat_tr.shape[1]}")

    # ── LightGBM（元コードと同一パラメータ）──
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr, y_tr_aug,
        eval_set=[(feat_va, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va = np.expm1(lgb_model.predict(feat_va))
    p_te = np.expm1(lgb_model.predict(feat_te))

    oof_lgb[va_idx] = p_va
    final_lgb += p_te / 5

    y_va_real = np.expm1(y_va)
    rmse = np.sqrt(mean_squared_error(y_va_real, p_va))
    fold_rmses.append(rmse)
    print(f"  🌟 LGB RMSE: {rmse:.4f}")

    # 特徴量重要度（Fold 0）
    if fold == 0:
        imp = lgb_model.feature_importances_
        n_snv = snv_tr.shape[1]
        n_d1 = d1_tr.shape[1]
        cat = {}
        pos = 0
        cat['SNV'] = np.sum(imp[pos:pos+n_snv]); pos += n_snv
        cat['d1'] = np.sum(imp[pos:pos+n_d1]); pos += n_d1
        cat['PCA'] = np.sum(imp[pos:pos+10]); pos += 10
        cat['KNN_mean'] = imp[pos]; pos += 1
        cat['ratio'] = imp[pos]; pos += 1
        cat['std'] = imp[pos]; pos += 1

        total = sum(cat.values())
        print(f"\n  📊 特徴量重要度:")
        for name, val in sorted(cat.items(), key=lambda x: -x[1]):
            pct = val / total * 100
            bar = '█' * int(pct / 2)
            print(f"     {name:15s}: {val:6.0f} ({pct:5.1f}%) {bar}")


# ============================================================
# 5. 全体評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_lgb))

print(f"  🌟 LGB OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# 樹種別
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_lgb[mask.values]
    rmse_s = np.sqrt(np.mean((y_s - p_s)**2))
    bias_s = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_s:7.2f} {bias_s:+7.2f}")


# ============================================================
# 6. 提出ファイル
# ============================================================
final_out = np.clip(final_lgb, 0, None)
submit[1] = final_out
out = f'submission_{pattern_name.replace(" ", "_")}.csv'
submit.to_csv(out, index=False, header=False)

print(f"\n✅ 提出ファイル: {out}")
print(f"📈 min={final_out.min():.1f}%, median={np.median(final_out):.1f}%, "
      f"max={final_out.max():.1f}%")

print(f"\n📌 全スコア比較:")
print(f"   LGB単独(元):       LB = 12.615 ← 現BEST")
print(f"   元Blend:           LB = 12.647")
print(f"   正則化強化:         LB = 12.760")
print(f"   Huber Loss:        LB = 12.770")
print(f"   PLS特徴量:          LB = 12.800")
print(f"   LGB+加重KNN:       LB = 12.940")
print(f"   LGB+d2:            LB = 13.410")
print(f"   今回({pattern_name}): LB = ???")

# ============================================================
# 7. Augmentationパラメータ感度分析（参考）
# ============================================================
print(f"\n{'='*60}")
print("📊 参考: Augmentationパラメータ候補")
print(f"{'='*60}")
print("""
  もし今回が改善した場合、次に試すパラメータ:
  
  n_augment:  300, 500, 1000  (多いほどデータ多様性↑、ノイズも↑)
  alpha:      0.1, 0.3, 0.5   (小さいほど元データに近い混合)
  noise_std:  0.001, 0.002, 0.005
  
  もし悪化した場合:
  → alpha を小さく (0.1) → 元データからあまり離れない混合
  → n_augment を減らす (200) → 元データの比率を維持
""")

🧪 テスト: Mixup
  → 異なる樹種のスペクトルを混合して疑似データ生成

───────────────────────────────────────────────────────
📁 Fold 1/5  (train:940, valid:270)
   検証樹種: ['ウエンジ', 'トチ']
   元データ: 940 samples
   拡張後:   1440 samples (+500)
   📐 LGB入力次元: 3123
  🌟 LGB RMSE: 7.9258

  📊 特徴量重要度:
     d1             :   1437 ( 64.0%) ████████████████████████████████
     SNV            :    602 ( 26.8%) █████████████
     KNN_mean       :    129 (  5.7%) ██
     PCA            :     61 (  2.7%) █
     ratio          :     15 (  0.7%) 
     std            :      1 (  0.0%) 

───────────────────────────────────────────────────────
📁 Fold 2/5  (train:981, valid:229)
   検証樹種: ['チェリー', 'ヒノキ']
  🌟 LGB RMSE: 17.8934

───────────────────────────────────────────────────────
📁 Fold 3/5  (train:1009, valid:201)
   検証樹種: ['ウォールナット', 'クリ']
  🌟 LGB RMSE: 20.7025

───────────────────────────────────────────────────────
📁 Fold 4/5  (train:959, valid:251)
   検証樹種: ['ナラ', 'ベイマツ', 'ホワイトオーク']
  🌟 LGB RMSE: 19.7955

──────────────────────────